<a href="https://colab.research.google.com/github/Phavouredphavour/Pizza-Sales-Analysis-/blob/main/curriculum/phase-2b-sql/weeks-01-08-teaching/week-02-groupby-aggregates-having/01-wednesday/lecture-materials/week-02-wed-demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — GROUP BY, Aggregates, HAVING: Summarising Data
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Compute summaries across groups with `GROUP BY`
- Use `SUM`, `COUNT`, `AVG`, `MIN`, and `MAX` to collapse many rows into one number per group
- Filter groups *after* aggregation with `HAVING`, and explain how it differs from `WHERE`

Last week you learned how to pull out and sort individual rows. Today you stop
looking at rows one at a time and start asking the database to *summarise* them for
you — which is where SQL starts doing real analytical work.

### Setup — run this cell first

Exactly the same setup cell as Week 1: it loads the 8 Olist tables into a file-based
SQLite database and points the `%%sql` cell magic at it. Run it once, top to bottom,
before any query below. If you restart the runtime, run it again.

In [1]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


Mounted at /content/drive
Searching your Google Drive for phase-2-python-sql.zip ...
Unzipping phase-2-python-sql.zip ...
Data folder: /content/olist_data/phase-2-python-sql
Loaded orders: 99,441 rows
Loaded customers: 99,441 rows
Loaded order_items: 112,650 rows
Loaded order_payments: 103,886 rows
Loaded order_reviews: 99,224 rows
Loaded products: 32,951 rows
Loaded sellers: 3,095 rows
Loaded product_category_translation: 71 rows

Database ready.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 482.8/482.8 kB 22.9 MB/s eta 0:00:00


## Why this matters

The `orders` table holds **99,441** rows. Last week we could count *one* slice at a
time — `WHERE order_status = 'delivered'` gave us **96,478**, and we would have had to
re-run the query, changing the status by hand, seven more times to see the whole
picture. That is fine for two statuses and unworkable for two hundred product
categories.

`GROUP BY` flips the question around: instead of "how many rows match *this* value?",
you ask "**for every distinct value in this column, how many rows are there?**" — and
the database answers all of them in a single pass. Combined with the aggregate
functions (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`) and `HAVING`, this is how one query
turns 99,441 raw rows into the handful of numbers a manager actually wants to see.

## 1. `GROUP BY` with `COUNT(*)`

`GROUP BY order_status` tells SQLite to gather all rows that share the same
`order_status` into one bucket, and then return **one row per bucket** instead of one
row per order. Whatever you put in `SELECT` alongside the grouped column must be an
*aggregate* — a function that collapses a whole bucket down to a single value.
`COUNT(*)` is the simplest one: it just counts how many rows landed in the bucket.

The mental model is a spreadsheet pivot table: the grouped column becomes the row
labels down the left, and the aggregate becomes the value column. Notice that we sort
by the alias `count` — you may order by a column you created in the `SELECT` list.

In [2]:
%%sql
-- One row per order_status, with how many orders fell into each bucket.
-- Expected (8 rows): delivered 96,478 | shipped 1,107 | canceled 625 |
--                    unavailable 609 | invoiced 314 | processing 301 |
--                    created 5 | approved 2
SELECT order_status, COUNT(*) AS count
FROM orders
GROUP BY order_status
ORDER BY count DESC

,order_status,count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


## 2. `SUM`, `AVG`, `MIN`, `MAX` — more than just counting

`COUNT(*)` answers "how many?". The other four aggregates answer "how much?":

- `SUM(col)` — adds every value in the bucket (total revenue, total freight)
- `AVG(col)` — the arithmetic mean of the bucket (typical order value)
- `MIN(col)` / `MAX(col)` — the smallest and largest value in the bucket (the range)

You can ask for several of them in the same `SELECT`, and they all respect the same
`GROUP BY`. `ROUND(x, 2)` keeps money readable at two decimal places instead of
printing fifteen digits of floating-point noise. Below, one query gives us the entire
payment picture: volume, typical size, total value, and the range — per payment
type.

In [3]:
%%sql
-- A full profile of every payment method in one query.
-- Expected: credit_card 76,795 tx, avg 163.32, total 12,542,084.19
--           boleto      19,784 tx, avg 145.03, total  2,869,361.27
--           voucher      5,775 tx, avg  65.70, total    379,436.87
--           debit_card   1,529 tx, avg 142.57, total    217,989.79
--           not_defined      3 tx, avg   0.00, total          0.00
SELECT payment_type,
       COUNT(*)                   AS transaction_count,
       ROUND(AVG(payment_value), 2) AS avg_value,
       ROUND(SUM(payment_value), 2) AS total_value,
       ROUND(MIN(payment_value), 2) AS min_value,
       ROUND(MAX(payment_value), 2) AS max_value
FROM order_payments
GROUP BY payment_type
ORDER BY transaction_count DESC;

,payment_type,transaction_count,avg_value,total_value,min_value,max_value
0,credit_card,76795,163.32,12542084.19,0.01,13664.08
1,boleto,19784,145.03,2869361.27,11.62,7274.88
2,voucher,5775,65.70,379436.87,0.00,3184.34
3,debit_card,1529,142.57,217989.79,13.38,4445.50
4,not_defined,3,0.00,0.00,0.00,0.00


## 3. Top-N groups — `GROUP BY` + `ORDER BY` + `LIMIT`

Grouping on a column with many distinct values (Brazil has 27 states) gives you a long
result. Nobody reads 27 rows looking for the biggest one, so you finish the query the
way you did last week: `ORDER BY` the aggregate descending, then `LIMIT` to the top
handful.

The clause order matters and is not negotiable: `FROM` → `GROUP BY` → `ORDER BY` →
`LIMIT`. SQLite builds the groups first, sorts the *grouped* result second, and only
then throws away everything past the limit — so `LIMIT 10` here means "the ten largest
states", not "look at only ten customers".

In [4]:
%%sql
-- Where are Olist's customers? Top 10 states by customer count.
-- Expected (top 5): SP 41,746 | RJ 12,852 | MG 11,635 | RS 5,466 | PR 5,045
SELECT customer_state, COUNT(*) AS order_count
FROM customers
GROUP BY customer_state
ORDER BY order_count DESC
LIMIT 10;

,customer_state,order_count
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


## 4. `HAVING` — filtering the groups themselves

`WHERE` and `HAVING` both filter, but they run at different moments and see different
things:

- **`WHERE` filters raw rows *before* the grouping happens.** It can only see columns
  that exist in the table.
- **`HAVING` filters whole groups *after* the aggregates are computed.** It can see
  `COUNT(*)`, `SUM(...)`, and friends — because by then they exist.

Think of it as a two-stage sieve: `WHERE` decides which rows are allowed into the
buckets, `HAVING` decides which finished buckets are worth reporting. Here we want to
ignore the marginal payment methods and keep only the ones carrying serious volume —
more than 5,000 transactions — which is a statement about the *group*, so it belongs
in `HAVING`.

In [5]:
%%sql
-- Only payment types with more than 5,000 transactions.
-- not_defined (3) and debit_card (1,529) are filtered out by HAVING.
-- Expected (3 rows): credit_card 76,795 | boleto 19,784 | voucher 5,775
SELECT payment_type, COUNT(*) AS count
FROM order_payments
GROUP BY payment_type
HAVING COUNT(*) > 5000
ORDER BY count DESC;

,payment_type,count
0,credit_card,76795
1,boleto,19784
2,voucher,5775


## Going deeper — turning group counts into percentages

A count on its own rarely settles an argument: "credit card: 76,795 transactions" only
means something once you know it out of how many. To turn a group count into a share
of the whole, divide it by a scalar subquery that counts *every* row in the table —
the subquery in the parentheses runs once and returns a single number.

Two traps live in that one line. First, **integer division truncates**: `COUNT(*)` and
the subquery are both integers, so `76795 / 103886` evaluates to `0`, not `0.739`. You
must force real division by writing `* 100.0` (rather than `* 100`) so at least one
operand is a REAL. Second, `ROUND(..., 1)` is what keeps the output readable —
without it you get a long float tail. Credit card comes out at **73.9%** of all
103,886 payment transactions, which is the headline fact about how Brazilians paid
Olist.

In [6]:
%%sql
-- WRONG (commented out) — integer division truncates every share to 0:
--   SELECT payment_type, COUNT(*) * 100 / (SELECT COUNT(*) FROM order_payments) AS pct
--   FROM order_payments GROUP BY payment_type
-- CORRECT — 100.0 forces REAL division:
SELECT payment_type,
       COUNT(*) AS transaction_count,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM order_payments), 1) AS pct_of_all
FROM order_payments
GROUP BY payment_type
ORDER BY transaction_count DESC   -- Expected: credit_card 76,795 = 73.9% of 103,886

,payment_type,transaction_count,pct_of_all
0,credit_card,76795,73.9
1,boleto,19784,19.0
2,voucher,5775,5.6
3,debit_card,1529,1.5
4,not_defined,3,0.0


## Common mistakes

**Mistake 1 — putting an aggregate in `WHERE`.** `WHERE COUNT(*) > 5000` looks
perfectly reasonable and fails immediately with
`sqlite3.OperationalError: misuse of aggregate function COUNT()`. `WHERE` runs before
any grouping, so at that moment `COUNT(*)` does not exist yet. Any condition about an
aggregate belongs in `HAVING`.

**Mistake 2 — selecting a column that is neither grouped nor aggregated.** If you
write `SELECT payment_type, payment_value, COUNT(*) ... GROUP BY payment_type`, most
databases refuse to run it — but **SQLite silently accepts it** and hands you the value
from an *arbitrary* row in each bucket. No error, no warning, and a number that looks
like data but means nothing. This is the single nastiest GROUP BY trap in SQLite. The
fix is to decide what you actually meant and say it: `MAX(payment_value)`,
`AVG(payment_value)`, or add the column to the `GROUP BY`.

In [7]:
%%sql
-- ── COMMON MISTAKE 1: aggregate in WHERE ────────────────────────────
-- WRONG — errors with "misuse of aggregate function COUNT()":
--   SELECT payment_type, COUNT(*) AS count
--   FROM order_payments
--   WHERE COUNT(*) > 5000
--   GROUP BY payment_type
-- CORRECT — filter the GROUPS with HAVING, and note that WHERE can still
-- filter the ROWS first (here: ignore zero-value payment records):
SELECT payment_type, COUNT(*) AS count
FROM order_payments
WHERE payment_value > 0          -- row filter, runs FIRST
GROUP BY payment_type
HAVING COUNT(*) > 5000           -- group filter, runs AFTER aggregation
ORDER BY count DESC

,payment_type,count
0,credit_card,76795
1,boleto,19784
2,voucher,5769


In [8]:
%%sql
-- ── COMMON MISTAKE 2: an ungrouped, unaggregated column ─────────────
-- WRONG — SQLite runs this without complaint but payment_value comes from
-- an ARBITRARY row in each group, so the number is meaningless:
--   SELECT payment_type, payment_value, COUNT(*) AS count
--   FROM order_payments
--   GROUP BY payment_type
-- CORRECT — say what you actually mean about the group:
SELECT payment_type,
       COUNT(*)                     AS count,
       ROUND(MAX(payment_value), 2) AS largest_payment,
       ROUND(AVG(payment_value), 2) AS avg_payment
FROM order_payments
GROUP BY payment_type
ORDER BY count DESC

,payment_type,count,largest_payment,avg_payment
0,credit_card,76795,13664.08,163.32
1,boleto,19784,7274.88,145.03
2,voucher,5775,3184.34,65.70
3,debit_card,1529,4445.50,142.57
4,not_defined,3,0.00,0.00


## Mini-challenge — your turn

⏱ ~5–10 minutes

Ops only care about order statuses that affect a meaningful number of customers.
Using `GROUP BY` and `HAVING` on the `orders` table, list **the order statuses with
more than 500 orders**, largest first.

**Expected:** 4 rows — `delivered` (96,478), `shipped` (1,107), `canceled` (625) and
`unavailable` (609). Everything from `invoiced` (314) down is filtered out.

Hint: it is the query from section 1 with one extra clause between `GROUP BY` and
`ORDER BY`.

In [10]:
%%sql
-- ⏱ ~5-10 min — your turn! Replace the placeholder below with your own query.
SELECT order_status, COUNT(*) AS todo
FROM orders
GROUP BY order_status
HAVING COUNT(*) > 500
ORDER BY todo DESC

,order_status,todo
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609


## Session Summary

| Clause / function | What it does | Example |
|---|---|---|
| `GROUP BY` | collapse rows into one row per distinct value | `GROUP BY order_status` |
| `COUNT(*)` | how many rows in the group | `SELECT order_status, COUNT(*) ...` |
| `SUM(col)` | total of a numeric column per group | `SUM(payment_value)` |
| `AVG(col)` | mean value per group | `AVG(payment_value)` |
| `MIN(col)` / `MAX(col)` | smallest / largest value per group | `MAX(payment_value)` |
| `ROUND(x, 2)` | keep money to 2 decimal places | `ROUND(AVG(payment_value), 2)` |
| `HAVING` | filter **groups** after aggregation | `HAVING COUNT(*) > 5000` |
| `WHERE` (with `GROUP BY`) | filter **rows** before aggregation | `WHERE payment_value > 0` |
| `ORDER BY` an alias | sort by a computed aggregate | `ORDER BY count DESC` |
| `* 1.0` / `* 100.0` | force REAL division for shares | `COUNT(*) * 100.0 / total` |

**Clause order to memorise:** `SELECT` … `FROM` … `WHERE` … `GROUP BY` … `HAVING` …
`ORDER BY` … `LIMIT`.

---
**Coming up Thursday**: we point today's tools at the `order_reviews` table to build a
review-score distribution — counts and percentages per star rating — and compare it
against the single overall average score. You'll see why an average of 4.09 can hide a
very lopsided distribution, then work through a set of group exercises on items,
installments, freight, and customer states.